# SmartHire GenAI — 04: Embedding Model Comparison
Compares Gemini API dense embeddings against a TF-IDF sparse baseline for job-matching retrieval quality.

In [ ]:
import sys
sys.path.append('..')
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine
from src.search.embed import EmbeddingManager

# Test corpus: job descriptions
job_corpus = [
    "Machine Learning Engineer: Design and deploy ML models using PyTorch, TensorFlow, and MLflow for production inference pipelines.",
    "Senior Backend Developer: Build scalable REST APIs with Python, FastAPI, PostgreSQL, Docker, and Kubernetes.",
    "Data Analyst: Analyze business data using SQL, Tableau, Power BI, and Excel to generate actionable insights.",
    "Cloud DevOps Engineer: Manage CI/CD pipelines, AWS infrastructure, Terraform, and container orchestration.",
    "Cybersecurity Analyst: Monitor SIEM alerts, conduct incident response, and perform vulnerability assessments.",
]

# Test queries: candidate profiles
test_queries = [
    {"query": "Python developer experienced in PyTorch, deep learning, and model deployment", "expected_best": 0},
    {"query": "Backend engineer skilled in FastAPI, Docker, and PostgreSQL databases", "expected_best": 1},
    {"query": "Business analyst proficient in SQL, Excel, and data visualization with Tableau", "expected_best": 2},
    {"query": "DevOps specialist with AWS, Terraform, and Kubernetes experience", "expected_best": 3},
    {"query": "Security professional experienced in SIEM monitoring and incident response", "expected_best": 4},
]

print(f"Corpus: {len(job_corpus)} jobs | Queries: {len(test_queries)} candidates")

In [ ]:
# --- Model A: Gemini Dense Embeddings (3072-dim) ---
embed_mgr = EmbeddingManager()
gemini_corpus_vecs = embed_mgr.embed_texts(job_corpus)
gemini_query_vecs = embed_mgr.embed_texts([q["query"] for q in test_queries])

gemini_sims = sklearn_cosine(gemini_query_vecs, gemini_corpus_vecs)
print("Gemini Embedding Similarity Matrix:")
print(np.round(gemini_sims, 3))

# --- Model B: TF-IDF Sparse Baseline ---
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_corpus_vecs = tfidf.fit_transform(job_corpus)
tfidf_query_vecs = tfidf.transform([q["query"] for q in test_queries])

tfidf_sims = sklearn_cosine(tfidf_query_vecs, tfidf_corpus_vecs).toarray() if hasattr(sklearn_cosine(tfidf_query_vecs, tfidf_corpus_vecs), 'toarray') else sklearn_cosine(tfidf_query_vecs, tfidf_corpus_vecs)
print("\nTF-IDF Baseline Similarity Matrix:")
print(np.round(tfidf_sims, 3))

In [ ]:
def evaluate_hit_at_1(sim_matrix, test_queries):
    """Compute Hit@1 accuracy: does the top-ranked result match the expected best job?"""
    hits = 0
    results = []
    for i, tq in enumerate(test_queries):
        predicted_best = int(np.argmax(sim_matrix[i]))
        is_hit = predicted_best == tq["expected_best"]
        if is_hit:
            hits += 1
        results.append({
            "query": tq["query"][:60],
            "expected": job_corpus[tq["expected_best"]][:40],
            "predicted": job_corpus[predicted_best][:40],
            "score": float(sim_matrix[i][predicted_best]),
            "hit": is_hit,
        })
    accuracy = hits / len(test_queries)
    return accuracy, results

gemini_acc, gemini_details = evaluate_hit_at_1(gemini_sims, test_queries)
tfidf_acc, tfidf_details = evaluate_hit_at_1(tfidf_sims, test_queries)

print(f"{'Model':<25} {'Hit@1 Accuracy':>15}")
print(f"{'-'*40}")
print(f"{'Gemini embedding-001':<25} {gemini_acc*100:>14.1f}%")
print(f"{'TF-IDF Baseline':<25} {tfidf_acc*100:>14.1f}%")
print()

print("\n--- Gemini Detailed Results ---")
for r in gemini_details:
    print(f"  {'✅' if r['hit'] else '❌'} {r['query']} → {r['predicted']} (score: {r['score']:.3f})")

print("\n--- TF-IDF Detailed Results ---")
for r in tfidf_details:
    print(f"  {'✅' if r['hit'] else '❌'} {r['query']} → {r['predicted']} (score: {r['score']:.3f})")

print(f"\n📊 Conclusion: Gemini dense embeddings achieve {gemini_acc*100:.0f}% Hit@1 vs TF-IDF baseline {tfidf_acc*100:.0f}% Hit@1.")
if gemini_acc > tfidf_acc:
    print("   → Gemini embeddings capture semantic similarity beyond lexical overlap, justifying API cost.")
elif gemini_acc == tfidf_acc:
    print("   → Both models achieve equal accuracy on this small benchmark. Gemini offers richer semantic understanding for diverse queries.")
else:
    print("   → TF-IDF performs competitively on this keyword-heavy benchmark. Gemini excels on paraphrased/conceptual queries.")